In [1]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import numpy as np
import time



In [2]:
#First we scrape the website
URL = "https://amharic.voanews.com/"
response =  requests.get(URL)

soup = BeautifulSoup(response.text,'html.parser')



In [4]:
categories = []
for h2 in soup.find_all("h2",class_ = "section-head"):
    
    a = h2.find("a")

    if a and a["href"].startswith("/") and not "z"  in a["href"]:
        category_name = a.get_text(strip=True)
        category_url = "https://amharic.voanews.com" + a["href"]
        categories.append({
            "name": category_name,
            "url": category_url
        })



print(f"Found {len(categories)} links")
print(categories)

Found 9 links
[{'name': 'ኢትዮጵያ/ኤርትራ', 'url': 'https://amharic.voanews.com/ethiopia-eritrea-news'}, {'name': 'አፍሪካ', 'url': 'https://amharic.voanews.com/africa-news'}, {'name': 'አሜሪካ', 'url': 'https://amharic.voanews.com/us-news'}, {'name': 'መካከለኛው ምሥራቅ', 'url': 'https://amharic.voanews.com/middle-east'}, {'name': 'ዓለም አቀፍ', 'url': 'https://amharic.voanews.com/world-news'}, {'name': 'የእሷ ድምፅ', 'url': 'https://amharic.voanews.com/her-voice-'}, {'name': 'ኑሮ በጤንነት', 'url': 'https://amharic.voanews.com/healthy-living-show'}, {'name': 'ሃኪምዎን ይጠይቁ', 'url': 'https://amharic.voanews.com/ask-the-doctor'}, {'name': 'የፎቶ መድብል', 'url': 'https://amharic.voanews.com/photo-gallery'}]


In [5]:
category_links = []
for data in categories:
    url = data["url"]
    try:
        response = requests.get(url)

        soup = BeautifulSoup(response.text,"html.parser")
        
        title = ""
        sub_article = soup.find("a",class_="link-more")
        h1 = soup.find("h1")

        if h1 and sub_article:
            title = h1.get_text(strip = True)
            category_links.append({"title":title,"url":"https://amharic.voanews.com" + sub_article["href"]})
        else:
            print("Discarded Links",sub_article["href"])

    except:
        print()
print(category_links)





[{'title': 'ኢትዮጵያ/ኤርትራ', 'url': 'https://amharic.voanews.com/z/3661'}, {'title': 'አፍሪካ', 'url': 'https://amharic.voanews.com/z/3169'}, {'title': 'አሜሪካ', 'url': 'https://amharic.voanews.com/z/7511'}, {'title': 'መካከለኛው ምሥራቅ', 'url': 'https://amharic.voanews.com/z/5281'}, {'title': 'ዓለምአቀፍ', 'url': 'https://amharic.voanews.com/z/3737'}, {'title': 'የፎቶ መድብሎች', 'url': 'https://amharic.voanews.com/z/3174'}]


In [6]:
news_in_categories = []
all_news = []
for data in category_links:
    category_name = data["title"]
    url = data["url"]

    try:
        response = requests.get(url)
        soup = BeautifulSoup(response.text,"html.parser")

        cards = soup.find_all("li" , class_="mb-grid")
    
        for card in cards:
            h4 = card.find("h4") 
            titles= h4.get_text(strip=True) if h4 else None
            read_more = card.find("a" , href=True)["href"] if a else None
                     # make relative urls absolute
            if read_more and read_more.startswith("/"):
                read_more = "https://amharic.voanews.com" + read_more
            # date
            date = card.find("span", class_="date")
            date_posted = date.get_text(strip=True) if date else None

            # image
            img = card.find("img")
            image_url = img["src"] if img and img.has_attr("src") else None

            all_news.append({
                "category":category_name,
                "title": titles,
                "read_more_url": read_more,
                "date_posted": date_posted,
                "image_url": image_url
            })

        
           




        # h1 = soup.find("h1").get_text()
        # print(h1)
        # h4 = soup.find_all("h4")
        # titles = [item.get_text(strip=True) for item in h4]
        # print(titles)
        # read_details = [details.find("a")["href"] for details in h4 if details.find("a") ]
        # print(read_details)
        # date_posted = soup.find_all("span",class_="date")
        # print(date_posted)
        # thumb_divs = soup.find_all("div", class_="thumb")
        # img_url = [div.find("img")["src"] for div in thumb_divs if div.find("img")]
        # print(img_url)
        
    except:
        print("error")


df = pd.DataFrame(all_news)


In [8]:
df = pd.DataFrame(all_news)


# Step 2: Clean and save
print(df.isna().sum())

df['title'] = df['title'].str.strip()
df['date_posted'] = df['date_posted'].str.strip()

df.to_csv('ethiopia_news.csv', index=False, encoding='utf-8-sig')
print("File saaved succesfully: 'ethiopia_news.csv',")


category         0
title            0
read_more_url    0
date_posted      0
image_url        0
dtype: int64
File saaved succesfully: 'ethiopia_news.csv',
